# Donor retention — CRISP-DM walkthrough

Run cells **top to bottom** for a clean video narrative.

**Six phases (one sentence each):**

1. **Business understanding** — What decision are we supporting?  
2. **Data understanding** — What does the data look like (size, target, gaps)?  
3. **Data preparation** — What goes into the model, and how do we clean it?  
4. **Modeling** — What algorithm do we train?  
5. **Evaluation** — How good is it, and are we overfitting?  
6. **Deployment** — How do we save and use the model?

This notebook walks through **all six CRISP-DM phases** in order (Phase 6 saves the model and prints a short deployment / quality summary).

## Phase 1: Business understanding

**Decision:** Should we flag supporters who are **likely to lapse** so we can reach out?

**Target:** `is_retained` — 1 if they gave in the last 365 days, 0 if lapsed.

**Inputs:** RFM-style features already rolled up to **one row per supporter** in `donor_and_potential_growth.csv`.

In [1]:
# Quick peek at the engineered table (one row per supporter)
from pathlib import Path

import pandas as pd

_repo = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "Dataset").is_dir()),
    None,
)
if _repo is None:
    raise FileNotFoundError("Could not find Dataset/ — open Jupyter from inside the ml-pipelines repo.")

df = pd.read_csv(
    _repo / "Dataset" / "lighthouse_csv_v7" / "Created .csv for Pipelines" / "donor_and_potential_growth.csv"
)
df.head()

,supporter_id,supporter_type,relationship_type,region,acquisition_channel,status,recency_days,frequency,total_monetary_value,avg_monetary_value,is_recurring_donor,social_referral_count,top_program_interest,is_retained
0,1,SocialMediaAdvocate,Local,Luzon,SocialMedia,Active,10.0,12.0,9000.03,750.002500,1,3.0,Wellbeing,1
1,2,Volunteer,Local,Mindanao,SocialMedia,Active,297.0,4.0,3877.36,969.340000,0,1.0,Education,1
2,3,MonetaryDonor,Local,Luzon,SocialMedia,Active,169.0,16.0,12448.13,778.008125,1,1.0,Maintenance,1
3,4,MonetaryDonor,PartnerOrganization,Mindanao,Church,Active,0.0,11.0,9934.62,903.147273,1,3.0,Wellbeing,1
4,5,InKindDonor,PartnerOrganization,Mindanao,Website,Active,150.0,5.0,4751.17,950.234000,0,1.0,Transport,1


## Phase 2: Data understanding

**What we do here:** Confirm the data matches expectations, look at the target, and spot missing values.

**Talking points for video:** 420 gift rows roll up to 60 supporters. Most rows are retained (`is_retained` = 1), so accuracy alone can be misleading later — use stratified CV and check performance on the **lapsed** class. One person has no gifts and missing fields; we fix that in Phase 3.

In [2]:
# Phase 2 — row counts, missing values, target balance
import numpy as np
from pathlib import Path

_repo = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "Dataset").is_dir()),
    None,
)
if _repo is None:
    raise FileNotFoundError("Could not find Dataset/ — open Jupyter from inside the ml-pipelines repo.")
base = _repo / "Dataset" / "lighthouse_csv_v7"
eng = base / "Created .csv for Pipelines" / "donor_and_potential_growth.csv"

df = pd.read_csv(eng)
donations = pd.read_csv(base / "donations.csv")
supporters = pd.read_csv(base / "supporters.csv")

print("Manifest:", len(donations), "donations,", len(supporters), "supporters,", len(df), "engineered rows")
print("Duplicate supporter_id:", df["supporter_id"].duplicated().sum())
print("\nMissing values:\n", df.isnull().sum()[lambda s: s > 0])
print("\nis_retained:\n", df["is_retained"].value_counts().sort_index())
print(f"Retained share: {df['is_retained'].mean():.3f}")

num = [
    "recency_days", "frequency", "total_monetary_value", "avg_monetary_value",
    "social_referral_count", "is_recurring_donor",
]
display(df[num].describe().T)
print("corr(total, avg gift) =", df["total_monetary_value"].corr(df["avg_monetary_value"]))
print("\ntop_program_interest:\n", df["top_program_interest"].value_counts(dropna=False))

Manifest: 420 donations, 60 supporters, 60 engineered rows
Duplicate supporter_id: 0

Missing values:
 recency_days            1
avg_monetary_value      1
top_program_interest    1
dtype: int64

is_retained:
 is_retained
0     9
1    51
Name: count, dtype: int64
Retained share: 0.850


,count,mean,std,min,25%,50%,75%,max
recency_days,59.0,185.542373,186.852452,0.00,59.500000,118.000000,267.500000,797.00
frequency,60.0,7.000000,4.569612,0.00,4.000000,6.000000,9.000000,23.00
total_monetary_value,60.0,4895.130167,3571.012801,0.00,2161.717500,3926.685000,6871.930000,14240.29
avg_monetary_value,59.0,702.593281,375.783054,27.12,464.234444,676.773333,846.219059,2356.92
social_referral_count,60.0,1.283333,1.222552,0.00,0.000000,1.000000,2.000000,5.00
is_recurring_donor,60.0,0.300000,0.462125,0.00,0.000000,0.000000,1.000000,1.00


corr(total, avg gift) = 0.4498690864053522

top_program_interest:
 top_program_interest
Operations     20
Education      19
Wellbeing      10
Maintenance     4
Transport       3
Outreach        3
NaN             1
Name: count, dtype: int64


## Phase 3: Data preparation

**What we do here:** Pick the columns the model sees, fix the one bad row, then scale numbers and encode the program category.

**Talking points for video:** Seven features feed the model; `is_retained` is the label. Cleaning and the sklearn preprocessor are defined in the **next code cell** (same notebook). When you train for real, fit the scaler **inside** cross-validation so holdout rows do not leak into preprocessing.

In [3]:
# Phase 3 — feature schema, cleaning, preprocessor, then X and y (all in-notebook)
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = [
    "recency_days",
    "frequency",
    "total_monetary_value",
    "avg_monetary_value",
    "social_referral_count",
    "is_recurring_donor",
]
CATEGORICAL_FEATURES = ["top_program_interest"]
TARGET_RETENTION = "is_retained"
TARGET_GROWTH = "total_monetary_value"


def clean_engineered_donor_df(df):
    out = df.copy()
    max_recency = out["recency_days"].max(skipna=True)
    if pd.isna(max_recency):
        max_recency = 0.0
    out.loc[out["recency_days"].isna(), "recency_days"] = float(max_recency)

    zero_gift = out["frequency"].fillna(0) == 0
    out.loc[out["avg_monetary_value"].isna() & zero_gift, "avg_monetary_value"] = 0.0
    med_avg = out["avg_monetary_value"].median(skipna=True)
    if pd.isna(med_avg):
        med_avg = 0.0
    out["avg_monetary_value"] = out["avg_monetary_value"].fillna(med_avg)

    out["social_referral_count"] = out["social_referral_count"].fillna(0.0)

    out["top_program_interest"] = (
        out["top_program_interest"].fillna("Unknown").astype(str).str.strip()
    )
    out.loc[out["top_program_interest"] == "", "top_program_interest"] = "Unknown"

    return out


def assert_modeling_frame_ready(df):
    need = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    missing_cols = [c for c in need if c not in df.columns]
    assert not missing_cols, f"Missing columns: {missing_cols}"
    nulls = df[need].isnull().sum()
    bad = nulls[nulls > 0]
    assert bad.empty, f"Nulls remain after clean:\n{bad}"


def build_feature_preprocessor():
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUMERIC_FEATURES),
            ("cat", categorical_pipe, CATEGORICAL_FEATURES),
        ]
    )


_repo = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "Dataset").is_dir()),
    None,
)
if _repo is None:
    raise FileNotFoundError("Could not find Dataset/ — open Jupyter from inside the ml-pipelines repo.")
raw = pd.read_csv(
    _repo / "Dataset" / "lighthouse_csv_v7" / "Created .csv for Pipelines" / "donor_and_potential_growth.csv"
)
df_clean = clean_engineered_donor_df(raw)
assert_modeling_frame_ready(df_clean)

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = df_clean[feature_cols]
y_retention = df_clean[TARGET_RETENTION]

print("Rows:", len(df_clean), "| Features:", feature_cols)
print("Nulls in X:", int(X.isnull().sum().sum()))
display(df_clean.loc[raw.isnull().any(axis=1), ["supporter_id"] + feature_cols])

preprocessor = build_feature_preprocessor()
X_prep = preprocessor.fit_transform(X)
print("Design matrix shape (exploratory fit on full data):", X_prep.shape)

Rows: 60 | Features: ['recency_days', 'frequency', 'total_monetary_value', 'avg_monetary_value', 'social_referral_count', 'is_recurring_donor', 'top_program_interest']
Nulls in X: 0


,supporter_id,recency_days,frequency,total_monetary_value,avg_monetary_value,social_referral_count,is_recurring_donor,top_program_interest
27,28,797.0,0.0,0.0,0.0,0.0,0,Unknown


Design matrix shape (exploratory fit on full data): (60, 13)


## Phase 4: Modeling

**What we do here:** Try **several classifiers** behind the same preprocessing `Pipeline`. Each model is scored with the **same stratified 5-fold CV** (no leakage: the pipeline refits on training folds only). We **keep the model with the highest mean CV F1 macro** (balances both classes), then fit that winner on all rows for Phase 6.

**Talking points:** F1 macro is a reasonable pick for imbalanced retention; metrics will still jump around with only 60 rows. Compare the printed **leaderboard** in the video before showing the winner.

In [4]:
# Phase 4 — compare models (same CV, same Pipeline prep) → pick best by CV F1 macro
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"acc": "accuracy", "f1_macro": "f1_macro", "roc_auc": "roc_auc"}

candidate_classifiers = {
    "random_forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
    ),
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_depth=4,
        max_iter=150,
        learning_rate=0.08,
        class_weight="balanced",
        random_state=42,
    ),
    "svc_rbf": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42,
    ),
}

rows = []
cv_by_name = {}
for name, clf in candidate_classifiers.items():
    pipe = Pipeline(
        steps=[("prep", build_feature_preprocessor()), ("clf", clone(clf))]
    )
    scores = cross_validate(
        pipe,
        X,
        y_retention,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
    )
    cv_by_name[name] = scores
    rows.append(
        {
            "model": name,
            "cv_acc_mean": scores["test_acc"].mean(),
            "cv_acc_std": scores["test_acc"].std(),
            "cv_f1_macro_mean": scores["test_f1_macro"].mean(),
            "cv_f1_macro_std": scores["test_f1_macro"].std(),
            "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
            "cv_roc_auc_std": scores["test_roc_auc"].std(),
        }
    )

leaderboard = pd.DataFrame(rows).sort_values(
    ["cv_f1_macro_mean", "cv_roc_auc_mean"], ascending=False
)
print("=== CV leaderboard (higher F1 macro is better; tie-break ROC-AUC) ===")
display(leaderboard)

best_name = leaderboard.iloc[0]["model"]
cv_scores = cv_by_name[best_name]
retention_model_name = best_name
retention_pipeline = Pipeline(
    steps=[
        ("prep", build_feature_preprocessor()),
        ("clf", clone(candidate_classifiers[best_name])),
    ]
)

print(f"\nSelected model: {best_name}")
print("(Selection uses mean CV F1 macro only; no peeking at held-out data beyond CV folds.)\n")
for key, label in [
    ("acc", "accuracy"),
    ("f1_macro", "F1 macro"),
    ("roc_auc", "ROC-AUC"),
]:
    test_m, test_s = cv_scores[f"test_{key}"].mean(), cv_scores[f"test_{key}"].std()
    train_m = cv_scores[f"train_{key}"].mean()
    print(f"{label}: CV test {test_m:.3f} (±{test_s:.3f}) | train {train_m:.3f}")

retention_pipeline.fit(X, y_retention)
print(f"\nFitted `retention_pipeline` ({best_name}) on all {len(X)} rows (Phase 6 save).")

=== CV leaderboard (higher F1 macro is better; tie-break ROC-AUC) ===


,model,cv_acc_mean,cv_acc_std,cv_f1_macro_mean,cv_f1_macro_std,cv_roc_auc_mean,cv_roc_auc_std
3,svc_rbf,0.983333,0.033333,0.961905,0.076190,1.000000,0.000000
1,logistic_regression,0.950000,0.040825,0.911378,0.076057,1.000000,0.000000
0,random_forest,0.950000,0.066667,0.852814,0.212360,0.980000,0.040000
2,hist_gradient_boosting,0.800000,0.100000,0.671566,0.157258,0.865909,0.071119



Selected model: svc_rbf
(Selection uses mean CV F1 macro only; no peeking at held-out data beyond CV folds.)

accuracy: CV test 0.983 (±0.033) | train 1.000
F1 macro: CV test 0.962 (±0.076) | train 1.000
ROC-AUC: CV test 1.000 (±0.000) | train 1.000

Fitted `retention_pipeline` (svc_rbf) on all 60 rows (Phase 6 save).


## Phase 5: Evaluation

**What we do here:** Score the model honestly using **out-of-fold** predictions from the same 5-fold CV (each row is predicted when it was in the validation fold). Compare that to **train** accuracy on the fully fitted pipeline to discuss overfitting. Inspect **feature importances** from the final forest (names match the preprocessed columns).

**Talking points:** With only 9 lapsed donors, confusion counts will bounce by fold—focus on **recall for class 0** (catching lapses) and the **train vs CV** gap, not a single flashy accuracy number.

In [5]:
# Phase 5 — requires Phase 4: retention_pipeline, X, y_retention, cv, cv_scores
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import cross_val_predict

y_pred_cv = cross_val_predict(retention_pipeline, X, y_retention, cv=cv)

print("=== Confusion matrix (rows = true, cols = pred) — out-of-fold ===")
print("Labels: 0 = Lapsed, 1 = Retained\n")
print(confusion_matrix(y_retention, y_pred_cv))
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(
    y_retention, y_pred_cv, display_labels=["Lapsed", "Retained"], ax=ax
)
ax.set_title("Retention — out-of-fold predictions")
plt.tight_layout()
plt.show()

print("\n=== Classification report (out-of-fold) ===")
print(
    classification_report(
        y_retention,
        y_pred_cv,
        target_names=["Lapsed (0)", "Retained (1)"],
        digits=3,
    )
)

y_train_hat = retention_pipeline.predict(X)
print("=== Overfitting check ===")
print(f"Train accuracy (fit on all rows): {accuracy_score(y_retention, y_train_hat):.3f}")
print(
    f"CV test accuracy (mean ± std):    {cv_scores['test_acc'].mean():.3f} ± {cv_scores['test_acc'].std():.3f}"
)

prep = retention_pipeline.named_steps["prep"]
clf = retention_pipeline.named_steps["clf"]
feat_names = prep.get_feature_names_out()

print("\n=== Feature signal (final model, preprocessed names) ===")
if hasattr(clf, "feature_importances_"):
    imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)
    display(imp.to_frame("importance"))
elif hasattr(clf, "coef_"):
    imp = pd.Series(np.abs(clf.coef_).ravel(), index=feat_names).sort_values(ascending=False)
    display(imp.to_frame("abs_coef"))
else:
    print(f"No importance/coef_ for estimator type: {type(clf).__name__}")

ModuleNotFoundError: No module named 'matplotlib'

## Phase 6: Deployment

**What we do here:** Save the **fitted** `retention_pipeline` to **`pipelines/retention_pipeline_v1.sav`** (used by the FastAPI app). The next cell smoke-tests a reload and prints a plain-language summary: what the artifact is, how strong the CV evidence is, and whether train vs CV suggests overfitting.

**New data:** Clean new rows with the same rules as Phase 3, build `X` with the same seven columns, then `loaded.predict(X_new)` or `predict_proba` for churn risk (e.g. probability of class 0).

In [ ]:
# Phase 6 — save artifact + deployment / quality summary
# Requires Phases 3–4 (and uses cv_scores from Phase 4; reuses same cv as Phase 5)
import joblib
from pathlib import Path

from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import cross_val_predict

_repo = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "Dataset").is_dir()),
    None,
)
if _repo is None:
    raise FileNotFoundError("Could not find Dataset/ — open Jupyter from inside the ml-pipelines repo.")
ARTIFACT_PATH = _repo / "pipelines" / "retention_pipeline_v1.sav"
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(retention_pipeline, ARTIFACT_PATH)
loaded = joblib.load(ARTIFACT_PATH)
_check = (loaded.predict(X) == retention_pipeline.predict(X)).all()
print(f"Saved pipeline to: {ARTIFACT_PATH}")
print(f"Reload check (predictions match): {_check}\n")

print("=" * 60)
print("PHASE 6 — WHAT THIS ARTIFACT IS")
print("=" * 60)
print(
    f"• One sklearn Pipeline: ColumnTransformer (impute + scale + one-hot)\n"
    f"  followed by **{retention_model_name}**.\n"
    "• Fitted on ALL 60 supporters after model comparison in Phase 4.\n"
    "• Use for scoring new supporters after Phase-3-style cleaning.\n"
)

y_oof = cross_val_predict(retention_pipeline, X, y_retention, cv=cv)
train_acc = accuracy_score(y_retention, retention_pipeline.predict(X))
cv_acc_m, cv_acc_s = cv_scores["test_acc"].mean(), cv_scores["test_acc"].std()
cv_f1_m = cv_scores["test_f1_macro"].mean()
cv_auc_m = cv_scores["test_roc_auc"].mean()
rec_lapsed = recall_score(y_retention, y_oof, pos_label=0, zero_division=0)
rec_retained = recall_score(y_retention, y_oof, pos_label=1, zero_division=0)

print("=" * 60)
print("MODEL QUALITY (honest estimate = cross-validation)")
print("=" * 60)
print(
    f"• CV accuracy (mean ± std):     {cv_acc_m:.3f} ± {cv_acc_s:.3f}\n"
    f"• CV F1 macro (mean):           {cv_f1_m:.3f}\n"
    f"• CV ROC-AUC (mean):            {cv_auc_m:.3f}\n"
    f"• Out-of-fold recall — Lapsed:  {rec_lapsed:.3f}  (catching people who truly lapsed)\n"
    f"• Out-of-fold recall — Retained:{rec_retained:.3f}\n"
)
print("Interpretation: With only 9 lapsed rows, these numbers move a lot by fold.")
print("Use them as directional, not as a guarantee on future donors.\n")

gap = train_acc - cv_acc_m
print("=" * 60)
print("OVERFITTING SIGNAL (train on all rows vs CV)")
print("=" * 60)
print(f"• Accuracy on training data (all 60 rows): {train_acc:.3f}")
print(f"• CV test accuracy (mean):                  {cv_acc_m:.3f}")
print(f"• Gap (train − CV mean):                    {gap:+.3f}")
if gap > 0.08:
    risk = "High — model fits this specific table much better than unseen folds suggest."
elif gap > 0.03:
    risk = "Moderate — some memorization is likely given N=60 and flexible trees."
else:
    risk = "Relatively small gap, but with 60 rows and many features after one-hot, stay cautious."
print(f"• Likelihood of overfitting: {risk}\n")

print("=" * 60)
print("CHURN-STYLE SCORE (example on training rows)")
print("=" * 60)
proba = retention_pipeline.predict_proba(X)[:, 0]
ex = (
    df_clean[["supporter_id"]]
    .assign(p_lapsed=proba)
    .sort_values("p_lapsed", ascending=False)
    .head(5)
)
print("Top 5 estimated P(lapsed) on current data (for illustration):")
display(ex)

## Pipeline summary

This notebook builds an **end-to-end donor retention (lapse) classifier** from the engineered supporter table `donor_and_potential_growth.csv` (one row per supporter under `Dataset/lighthouse_csv_v7/Created .csv for Pipelines/`).

**Goal:** Predict **`is_retained`** (1 = gave in the last 365 days, 0 = lapsed) so outreach can target supporters at risk of lapsing.

**Flow:**

1. **Load & explore** — Joined manifest checks (donations, supporters, engineered rows), class balance, missingness, and simple numeric summaries.
2. **Prepare features** — Seven inputs: six numeric RFM-style fields plus `top_program_interest`. Rows are cleaned (imputation rules for the zero-gift edge case, `Unknown` program), then passed through a **`ColumnTransformer`**: median imputation + **`StandardScaler`** for numerics, most-frequent + **`OneHotEncoder`** for the program column.
3. **Model selection** — Four classifiers (random forest, logistic regression, histogram gradient boosting, RBF SVM) are compared with **stratified 5-fold CV** inside a single **`Pipeline`** so preprocessing refits only on training folds. The winner is chosen by **mean CV F1 macro** (tie-break: ROC-AUC), then refit on **all 60 rows**.
4. **Evaluation** — **Out-of-fold** predictions drive the confusion matrix, classification report, and train-vs-CV overfitting check; feature importances are shown when the final estimator exposes them (e.g. not for SVM).
5. **Deployment** — The fitted **`retention_pipeline`** is saved to **`pipelines/retention_pipeline_v1.sav`** for the FastAPI app, with a reload smoke test and a short quality narrative (CV metrics, lapse recall, churn-style `predict_proba` example).

**Using new data:** Apply the same cleaning as Phase 3, build `X` with the same seven columns, then **`predict`** / **`predict_proba`** on the loaded pipeline. With **N ≈ 60** and a **small lapsed class**, treat metrics as **directional**, not production guarantees.
